# Chapter 1: Manual LLM Inference From First Principles

This chapter manually explores the core steps of LLM inference using a small pretrained model.

## 1. Setup: GPU, PyTorch and the tokenizer

Before getting into the actual inference loop, we first need three things:

1. a GPU that PyTorch can access
2. Hugging Face Transformers to load the model/tokenizer
3. the tokenizer for the model we are going to use

For this chapter I am using **Google Colab with an NVIDIA T4 GPU** and **Qwen2.5-0.5B-Instruct**.

The model is intentionally small. The goal right now is not to benchmark a powerful model. It is to understand what an inference runtime is actually doing without hiding everything behind `model.generate()`.

---

### Checking that PyTorch can access the GPU

```python
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
```

**PyTorch** is the framework we will use to run the model and work with tensors.

`torch.cuda` is PyTorch's interface to NVIDIA GPUs through CUDA.

```python
torch.cuda.is_available()
```

checks whether PyTorch can actually access a CUDA-capable GPU.

On Colab this should return:

```text
True
```

Then:

```python
torch.cuda.get_device_name(0)
```

returns the name of GPU `0`, where `0` means the first GPU available to us.

For this notebook:

```text
Tesla T4
```

So at this point the stack is basically:

```text
Python
   ↓
PyTorch
   ↓
CUDA
   ↓
NVIDIA T4
```

Later, when we run the model, PyTorch will use CUDA kernels underneath to execute the actual computations on this GPU.

---

### Installing Hugging Face Transformers

```python
!pip install -q transformers
```

`transformers` is Hugging Face's library for working with pretrained Transformer models.

We are using Hugging Face because we do **not** want to spend time training an LLM from scratch. We want a pretrained model whose computation we can use while implementing the inference/runtime logic ourselves.

So Hugging Face will give us things like:

```text
tokenizer
pretrained model weights
model architecture
```

But we will manually handle things such as:

```text
prefill
decode
KV cache
sampling
request state
scheduling
```

as we progress.

---

## Loading the tokenizer

```python
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
```

Before an LLM can process text, the text has to be converted into **tokens**.

A tokenizer handles:

```text
raw text
   ↓
token pieces
   ↓
integer token IDs
```

For example, something like:

```text
"capital of france"
```

might become:

```text
"capital"    → 65063
" of"        → 315
" france"    → 47587
```

These integers are not embeddings and they do not contain meaning themselves.

They are basically IDs into the model's vocabulary.

---

### `AutoTokenizer`

```python
AutoTokenizer
```

is a Hugging Face helper that figures out which tokenizer implementation the selected model uses.

Instead of manually doing something like:

```text
load Qwen tokenizer implementation
load its vocabulary
load its merge rules
load its special tokens
...
```

we simply provide the model name:

```python
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
```

and:

```python
AutoTokenizer.from_pretrained(model_name)
```

downloads and loads the tokenizer configuration that belongs to that model.

This matters because the **tokenizer and model have to agree on the vocabulary**.

If Qwen says:

```text
token ID 65063 = "capital"
```

then the model's embedding matrix expects `65063` to mean exactly that token.

Using a completely different tokenizer would therefore give the model the wrong token IDs.

---

At this point we have not loaded the actual neural network yet.

We only have:

```text
text
   ↓
Qwen tokenizer
   ↓
token IDs
```


In [2]:
#implementation

import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [3]:
#implementation
!pip install -q transformers
from transformers import AutoTokenizer
model_name='Qwen/Qwen2.5-0.5B-Instruct'
tokenizer= AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

### What we got

The first two lines were:

```text
True
Tesla T4
```

This confirms that PyTorch can access CUDA and that our Colab runtime is using an **NVIDIA Tesla T4**.

So our GPU setup is working.

After that, Hugging Face downloaded the tokenizer files for Qwen2.5-0.5B-Instruct:

```text
config.json
tokenizer_config.json
vocab.json
merges.txt
tokenizer.json
```

The main thing to understand is that a tokenizer is not just one function sitting somewhere.

It needs actual files that define things like:

```text
which tokens exist
which integer ID belongs to each token
how smaller text pieces get merged into tokens
special tokens used by the model
```

The warning:

```text
Warning: You are sending unauthenticated requests to the HF Hub.
```

is not an error.

It only means we are downloading public files without logging into Hugging Face. For this notebook, that is fine.

At this point we have:

```text
GPU ready
+
PyTorch ready
+
Qwen tokenizer loaded
```

We still have **not loaded the actual LLM yet**.

Next we will give the tokenizer a sentence and inspect exactly how raw text becomes token IDs.


In [4]:
#implementation
prompt= "capital of france is paris"
inputs_as_list= tokenizer(prompt) #return tokenized input with attention mask
inputs_as_tensors= tokenizer(prompt, return_tensors='pt') #return_tensors="pt" means return the result as PyTorch tensors instead of normal Python lists.
print(inputs_as_list)
print(inputs_as_tensors)

#attention mask tells the model which token positions are real input and which are just padding.

{'input_ids': [65063, 315, 47587, 374, 40858], 'attention_mask': [1, 1, 1, 1, 1]}
{'input_ids': tensor([[65063,   315, 47587,   374, 40858]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}


## 2. Tokenizing our first prompt

Now that the tokenizer is loaded, let's actually give it some text.

```python
prompt = "capital of france is paris"

inputs_as_list = tokenizer(prompt)
inputs_as_tensors = tokenizer(prompt, return_tensors="pt")

print(inputs_as_list)
print(inputs_as_tensors)
```

We are doing the same tokenization twice here.

The first call:

```python
tokenizer(prompt)
```

returns normal Python lists.

The second:

```python
tokenizer(prompt, return_tensors="pt")
```

returns PyTorch tensors.

`"pt"` simply means **PyTorch**.

We will eventually feed tensors into the model, so the second version is the one we will actually use for inference.

---

### Output

```text
{
    'input_ids': [65063, 315, 47587, 374, 40858],
    'attention_mask': [1, 1, 1, 1, 1]
}
```

and:

```text
{
    'input_ids': tensor([[65063, 315, 47587, 374, 40858]]),
    'attention_mask': tensor([[1, 1, 1, 1, 1]])
}
```

So our text:

```text
capital of france is paris
```

has been converted into **5 token IDs**:

```text
[65063, 315, 47587, 374, 40858]
```

These numbers are just IDs from Qwen's vocabulary.

At this point we still don't know which ID corresponds to which part of the sentence. We'll inspect that next.

---

## What is the attention mask?

We also got:

```text
[1, 1, 1, 1, 1]
```

This is the **attention mask**.

For now, think of it as:

```text
1 = this is a real token
0 = this position is only padding
```

Why would we ever need padding?

Imagine we want to process two prompts together:

```text
Prompt 1: "hello world"
Prompt 2: "hello"
```

If the first prompt becomes 2 tokens and the second becomes only 1, we cannot directly put them into one rectangular tensor.

So we might pad the shorter one:

```text
Prompt 1 → [token, token]
Prompt 2 → [token, PAD]
```

Then the masks become:

```text
Prompt 1 → [1, 1]
Prompt 2 → [1, 0]
```

The `0` tells the model:

> this position was added only to make the shapes match. It is not part of the actual prompt.

Our current prompt has no padding, so every position is real:

```text
[1, 1, 1, 1, 1]
```

One important thing: this is **not the causal attention mask** that stops a token from looking into the future.

We will get to that separately.

---

Right now our pipeline has reached:

```text
raw text
   ↓
tokenizer
   ↓
token IDs
   ↓
PyTorch tensor
```

Next, let's decode each token ID individually and see exactly how Qwen split our sentence.


In [6]:
#seeing what each of the integer value in list represents

for token_id in inputs_as_list["input_ids"]:
    print(token_id, repr(tokenizer.decode([token_id])))

65063 'capital'
315 ' of'
47587 ' france'
374 ' is'
40858 ' paris'


## 3. Seeing what each token ID actually means

We now know that our prompt became:

```text
[65063, 315, 47587, 374, 40858]
```

But those numbers are not useful to us unless we know what text each one represents.

So let's decode them one by one:

```python
for token_id in inputs_as_list["input_ids"]:
    print(token_id, repr(tokenizer.decode([token_id])))
```

### Output

```text
65063 'capital'
315   ' of'
47587 ' france'
374   ' is'
40858 ' paris'
```

So Qwen split:

```text
capital of france is paris
```

into:

```text
"capital"
" of"
" france"
" is"
" paris"
```

and then mapped those token pieces to integer IDs:

```text
"capital"  → 65063
" of"      → 315
" france"  → 47587
" is"      → 374
" paris"   → 40858
```

One detail worth noticing is that tokens like:

```text
" of"
" france"
" is"
```

include the **space before the word**.

That space is actually part of the token.

This is why tokenization is not the same as simply splitting a sentence by spaces.

A tokenizer works with pieces that exist in its learned vocabulary, and those pieces can include:

* full words
* parts of words
* spaces
* punctuation
* special tokens

So at this point, our pipeline is:

```text
"capital of france is paris"
        ↓
tokenizer
        ↓
["capital", " of", " france", " is", " paris"]
        ↓
[65063, 315, 47587, 374, 40858]
```

The important thing to remember is that **these integers still do not contain meaning by themselves**.

`65063` does not mathematically mean `"capital"`.

It is just the vocabulary ID assigned to that token.

Next, we will load the actual model and see how one of these token IDs becomes a learned embedding vector.


In [7]:
from transformers import AutoModelForCausalLM

model= AutoModelForCausalLM.from_pretrained(model_name,torch_dtype=torch.float16).cuda()
model.eval()



[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

## 4. Loading the actual model

So far we only had the tokenizer.

We could convert:

```text
text → token IDs
```

but we still did not have the neural network that actually processes those IDs and predicts what comes next.

Now let's load Qwen itself.

```python
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16
).cuda()

model.eval()
```

---

### What is `AutoModelForCausalLM`?

`CausalLM` means **Causal Language Model**.

Very simply, it is a model trained to do:

```text
previous tokens
      ↓
predict the next token
```

For example:

```text
"The capital of France is"
              ↓
            " Paris"
```

It is called **causal** because while predicting a token, it can only use the tokens that came before it. It cannot look into the future.

`AutoModelForCausalLM` is another Hugging Face helper.

Just like `AutoTokenizer` figured out which tokenizer Qwen needs, this figures out which model implementation should be created for:

```python
"Qwen/Qwen2.5-0.5B-Instruct"
```

---

## `from_pretrained()`

```python
AutoModelForCausalLM.from_pretrained(model_name)
```

does two major things:

```text
create the Qwen model architecture
+
load Qwen's already-trained weights into it
```

Remember what **weights** are:

They are the millions of learned numbers stored throughout the model.

Things like:

```text
embedding weights
attention weights
MLP weights
normalization parameters
...
```

We are not initializing a fresh random Qwen.

We are downloading the numbers that were already learned during Qwen's training.

---

## Why `float16`?

```python
dtype=torch.float16
```

tells Hugging Face to load the weights as **16-bit floating-point numbers**.

A float is just a number that can contain decimals, for example:

```text
0.013
-0.427
1.82
```

`float16` uses **2 bytes per number**.

This matters because an LLM contains hundreds of millions or billions of these numbers.

Using fewer bytes means:

```text
less GPU memory
+
less memory movement
```

which is extremely important for inference.

You may see older code using:

```python
torch_dtype=torch.float16
```

but recent Transformers versions warn that `torch_dtype` is deprecated, so we use:

```python
dtype=torch.float16
```

instead.

---

## Moving the model to the GPU

```python
.cuda()
```

moves the model weights from CPU memory into our T4's GPU memory.

So now the model actually lives on the GPU.

Conceptually:

```text
Qwen weights
   ↓
GPU VRAM
   ↓
Tesla T4
```

When we later call:

```python
model(...)
```

the actual Transformer computation can run on the GPU.

---

## `model.eval()`

```python
model.eval()
```

puts the model into **evaluation/inference mode**.

Some neural-network layers behave differently during training and inference.

For example, **dropout** randomly disables some values during training to help prevent overfitting.

We don't want that while generating text.

So `eval()` tells PyTorch:

> We are using this model for inference, not training.

One important detail:

`model.eval()` does **not** disable gradient tracking.

Later we will separately use:

```python
torch.no_grad()
```

when we actually run inference.

---

# What got downloaded?

We saw:

```text
model.safetensors
```

This is the important one.

It contains Qwen's pretrained model weights.

The file was roughly:

```text
988 MB
```

for this small 0.5B model.

`safetensors` is simply a file format commonly used to store model weights.

We also downloaded:

```text
generation_config.json
```

which contains default settings related to text generation.

We are not going to rely heavily on those defaults because the whole point of this project is to build the generation logic ourselves.

---

# Looking at the model architecture

Hugging Face also printed the structure of the model:

```text
Qwen2ForCausalLM(
    ...
)
```

This is basically PyTorch showing us:

> Here are all the major pieces that make up this neural network.

We do **not** need to understand every line yet.

But there are a few things worth noticing.

---

### The embedding layer

```text
Embedding(151936, 896)
```

This means Qwen has an embedding weight matrix shaped:

```text
[151936, 896]
```

where:

```text
151936 = number of tokens in the vocabulary
896    = numbers used to represent each token
```

So when our tokenizer gave us:

```text
65063 = "capital"
```

the model can essentially do:

```text
embedding_matrix[65063]
```

and get a vector containing:

```text
896 numbers
```

We will prove this ourselves in the next section instead of just believing the architecture printout.

---

### 24 Transformer layers

We also see:

```text
(0-23): 24 x Qwen2DecoderLayer
```

So Qwen2.5-0.5B contains **24 Transformer decoder layers** stacked one after another.

Very roughly:

```text
embedding
   ↓
Transformer layer 1
   ↓
Transformer layer 2
   ↓
...
   ↓
Transformer layer 24
   ↓
next-token scores
```

A small historical comparison is useful here.

The original **Attention Is All You Need** Transformer from 2017 used:

```text
6 encoder layers
+
6 decoder layers
```

with a hidden size of:

```text
512
```

Qwen is different:

```text
decoder-only
24 decoder layers
hidden size = 896
```

Modern LLMs changed the original Transformer architecture quite a bit, but the core idea of stacking Transformer blocks is still there.

We will go into those differences only when they become relevant.

---

### Attention projections

Inside every layer we also see:

```text
q_proj
k_proj
v_proj
o_proj
```

These are the weight matrices used by self-attention to create:

```text
Q = Query
K = Key
V = Value
```

We are not going into them yet.

Later, when we inspect the KV cache, these exact `k_proj` and `v_proj` dimensions will become very important.

---

### The MLP

Every layer also contains:

```text
Qwen2MLP
```

This is the feed-forward/MLP part of the Transformer block.

Again, not important for our immediate goal, so we'll leave it alone for now.

---

### `lm_head`

At the very end we see:

```text
lm_head: Linear(
    in_features=896,
    out_features=151936
)
```

This is important.

Near the end of the model we have a representation containing:

```text
896 numbers
```

But the model has:

```text
151936 possible vocabulary tokens
```

So `lm_head` converts:

```text
896-dimensional hidden state
             ↓
151936 scores
```

One score for every possible next token.

Those scores are called **logits**.

We'll inspect them ourselves shortly.

---

So after this cell our pipeline has grown from:

```text
text
 ↓
tokenizer
 ↓
token IDs
```

to:

```text
text
 ↓
tokenizer
 ↓
token IDs
 ↓
Qwen neural network loaded on GPU
```

But we still haven't passed our tokens through the model.

Before doing that, we'll first look at the very first operation the model performs on our token IDs:

**the embedding lookup.**


In [8]:
input_ids= inputs_as_tensors['input_ids'].cuda()
embeddings=model.model.embed_tokens(input_ids)
print(embeddings.shape)

# 1 = batch size/sequence, 5 = number of tokens, 896 = 896 numbers per vector or token as embeddings

torch.Size([1, 5, 896])


In [9]:
print(embeddings[0,0,:10]) #first 10 values out of its 896-dimensional embedding of first token from first sequence

#One important distinction: because we loaded a pretrained Qwen model,
#these are already learned embedding weights.
#They are no longer the random initialization values from before training.

tensor([-0.0126,  0.0002, -0.0098,  0.0082, -0.0068,  0.0075,  0.0376,  0.0109,
        -0.0253,  0.0099], device='cuda:0', dtype=torch.float16,
       grad_fn=<SliceBackward0>)


In [10]:
token_id= inputs_as_tensors['input_ids'][0, 0].item()
print(token_id)
print(torch.equal(embeddings[0,0],model.model.embed_tokens.weight[token_id]))

# proving token value is just row 65063 in the embedding matrix

65063
True


## 5. From token IDs to embeddings

We currently have our prompt represented as integer token IDs:

```text
[65063, 315, 47587, 374, 40858]
```

But the Transformer does not directly perform attention on integers like `65063`.

Before the tokens enter the Transformer layers, every token ID is converted into a **vector of numbers called an embedding**.

So the next part of the pipeline is:

```text
token ID
   ↓
embedding lookup
   ↓
vector of numbers
```

Let's do that manually.

```python
input_ids = inputs_as_tensors["input_ids"].cuda()

embeddings = model.model.embed_tokens(input_ids)

print(embeddings.shape)
```

### Moving the token IDs to the GPU

```python
inputs_as_tensors["input_ids"]
```

contains our token IDs as a PyTorch tensor.

We then use:

```python
.cuda()
```

to move that tensor onto the T4 GPU.

This is important because our model weights are already on the GPU.

For the model to process the tokens efficiently, the input tensor and the model need to be on the same device.

---

## The embedding layer

This line does the actual embedding lookup:

```python
embeddings = model.model.embed_tokens(input_ids)
```

Earlier, when we printed the model architecture, we saw:

```text
Embedding(151936, 896)
```

This means the model has an embedding matrix with shape:

```text
[151936, 896]
```

Think of it as a giant table:

```text
token ID 0       → [896 numbers]
token ID 1       → [896 numbers]
token ID 2       → [896 numbers]
...
token ID 65063   → [896 numbers]
...
token ID 151935  → [896 numbers]
```

There is one row for every token in Qwen's vocabulary.

Our first token was:

```text
65063 = "capital"
```

So the embedding layer essentially does:

```text
go to row 65063
↓
take the 896 numbers stored there
```

---

## The output shape

We got:

```text
torch.Size([1, 5, 896])
```

The three dimensions mean:

```text
[batch size, number of tokens, embedding size]

[1, 5, 896]
```

So:

```text
1   = one prompt
5   = five tokens in the prompt
896 = 896 numbers representing each token
```

Visually:

```text
"capital" → [896 numbers]
" of"     → [896 numbers]
" france" → [896 numbers]
" is"     → [896 numbers]
" paris"  → [896 numbers]
```

Our original input was:

```text
5 integer IDs
```

and it has now become:

```text
5 vectors × 896 numbers
```

This `[1, 5, 896]` tensor is much closer to what the Transformer actually works with.

---

## Looking inside one embedding

Let's inspect only the first 10 values of the first token's embedding:

```python
print(embeddings[0, 0, :10])
```

The indexing means:

```text
[0, 0, :10]
 ↑  ↑   ↑
 |  |   first 10 values
 |  first token
 first prompt
```

We got:

```text
tensor([
    -0.0126,
     0.0002,
    -0.0098,
     0.0082,
    -0.0068,
     0.0075,
     0.0376,
     0.0109,
    -0.0253,
     0.0099
], device='cuda:0', dtype=torch.float16, ...)
```

These are only the first **10 values**.

There are actually:

```text
896 values
```

in the full embedding for `"capital"`.

A few things in the output are worth noticing.

### `device='cuda:0'`

The embedding tensor is sitting on our first GPU.

So the lookup happened using data on the T4.

### `dtype=torch.float16`

Each value is stored as a 16-bit floating-point number, the same datatype we used when loading the model.

### These values are already learned

This is important.

If we created a brand-new model before training, the embedding matrix would start from initialized values that are roughly random.

But we loaded:

```python
from_pretrained(...)
```

so these numbers are already the result of Qwen's training.

The embedding matrix is therefore part of Qwen's **trained model weights**.

---

## Proving that an embedding is just a row lookup

We don't have to take this on faith.

Let's verify it directly.

```python
token_id = inputs_as_tensors["input_ids"][0, 0].item()

print(token_id)

print(
    torch.equal(
        embeddings[0, 0],
        model.model.embed_tokens.weight[token_id]
    )
)
```

The first line:

```python
inputs_as_tensors["input_ids"][0, 0]
```

selects:

```text
first prompt
↓
first token
```

which gives us:

```text
65063
```

`.item()` simply converts the one-value PyTorch tensor into a normal Python number.

Our output was:

```text
65063
True
```

The interesting part is:

```text
True
```

We compared:

```python
embeddings[0, 0]
```

with:

```python
model.model.embed_tokens.weight[65063]
```

and PyTorch confirmed that they are exactly equal.

So we have now experimentally proven:

```text
"capital"
    ↓
tokenizer
    ↓
65063
    ↓
embedding matrix row 65063
    ↓
896-number vector
```

There is no complicated mathematical formula converting:

```text
65063 → embedding
```

`65063` is basically an index telling the model:

> fetch row 65063 from the embedding weight matrix.

---

## Embeddings are model weights

This is an important connection.

The embedding matrix:

```text
[151936, 896]
```

is itself one of the model's learned **weight matrices**.

So when people talk about an LLM having hundreds of millions or billions of parameters, some of those parameters are sitting right here in the embedding table.

For this embedding matrix alone:

```text
151936 × 896
≈ 136 million values
```

Those are roughly **136 million learned parameters** just for representing vocabulary tokens.

The model has many other weight matrices later for:

```text
attention
Q/K/V projections
MLPs
normalization
output projection
```

The full parameter count is the combination of all of them.

---

### Small comparison with the original Transformer

The original 2017 Transformer used:

```text
embedding / hidden size = 512
```

Our Qwen model uses:

```text
embedding / hidden size = 896
```

The basic idea is the same:

```text
token ID → learned vector
```

but modern models choose different widths and architectures.

---

At this point our pipeline is:

```text
raw text
   ↓
tokenizer
   ↓
token IDs
   ↓
embedding lookup
   ↓
[batch, tokens, hidden size]

[1, 5, 896]
```

We have now reached the numerical representation that enters the Transformer.

Next, instead of manually stopping at the embedding layer, we will pass the tokens through the **entire model for the first time** and inspect what comes out.


In [11]:
with torch.no_grad():
  outputs=model(input_ids)

  #torch.no_grad() tells pytorch that we are doing inference, so don’t store gradient information for training

In [12]:
print(type(outputs))
print(outputs.keys())

#outputs is not just one tensor. Hugging Face returns an object containing different things the model can expose.

#logits = raw scores for what the next token could be
#past_key_values = the KV cache created from the tokens we already processed
# CausalLMOutputWithPast means output from a next-token language model, plus cached K/V tensors from the past sequence

<class 'transformers.modeling_outputs.CausalLMOutputWithPast'>
odict_keys(['logits', 'past_key_values'])


In [13]:
print(outputs.logits.shape)
# 151936 = one score for every token in the vocabulary

torch.Size([1, 5, 151936])


In [14]:
last_token_logits = outputs.logits[:, -1, :]

print(last_token_logits.shape)

#The important part is: for generation, we only care about the last position. Why? Because your input was: capital of france is paris
#The last position represents: “given everything up to paris, what token should come next?
#so the code gives one sequence, one score for every possible next token.
# That means for our one sequence, the model gave 151,936 raw scores, one for every possible next token.

torch.Size([1, 151936])


## 6. Running the first full forward pass

Until now we stopped manually at the embedding layer.

Now we will finally pass our token IDs through the **entire Qwen model**.

```python
with torch.no_grad():
    outputs = model(input_ids)

print(type(outputs))
print(outputs.keys())
```

---

## Why `torch.no_grad()`?

PyTorch normally keeps extra information during a forward pass because it assumes we may want to train the model afterward.

During training, it needs that information to calculate **gradients** and update the model weights.

But here we are only doing inference.

We want:

```text
input
  ↓
model
  ↓
output
```

We are not changing any weights.

So:

```python
with torch.no_grad():
```

tells PyTorch:

> just run the forward pass; don't keep the extra information needed for training.

This saves memory and avoids unnecessary work.

---

## Calling the model

```python
outputs = model(input_ids)
```

This is the first time our tokens go through the full model.

Very roughly, Qwen now does:

```text
token IDs
   ↓
embedding lookup
   ↓
Transformer layer 1
   ↓
Transformer layer 2
   ↓
...
   ↓
Transformer layer 24
   ↓
lm_head
   ↓
output
```

Remember, `lm_head` is the final layer that converts the model's hidden representation into scores over the whole vocabulary.

---

## What did the model return?

We got:

```text
<class 'transformers.modeling_outputs.CausalLMOutputWithPast'>

odict_keys(['logits', 'past_key_values'])
```

Hugging Face does not return just one tensor here.

It gives us an output object containing multiple things.

For now, the two important ones are:

```text
logits
past_key_values
```

### `logits`

These are the model's raw scores for possible next tokens.

Higher score = the model currently prefers that token more.

They are **not probabilities yet**.

We will convert them into probabilities later.

### `past_key_values`

This is the **KV cache** created while processing our prompt.

It stores the Keys and Values calculated by the attention layers for tokens that have already been processed.

We will inspect this properly soon.

For now just remember:

```text
logits          → what token could come next?
past_key_values → cached information about tokens already processed
```

The name:

```text
CausalLMOutputWithPast
```

basically means:

> output from a causal language model, including information from the already processed sequence.

---

# Inspecting the logits

Let's check their shape.

```python
print(outputs.logits.shape)
```

Our output was:

```text
torch.Size([1, 5, 151936])
```

Read this as:

```text
[batch size, sequence length, vocabulary size]

[1, 5, 151936]
```

So:

```text
1       = one prompt
5       = five input tokens
151936  = one score for every token in Qwen's vocabulary
```

The interesting part is that the model produced **151,936 scores for every one of our five token positions**.

Conceptually:

```text
"capital" → 151,936 scores
" of"     → 151,936 scores
" france" → 151,936 scores
" is"     → 151,936 scores
" paris"  → 151,936 scores
```

Why?

Because during training, a causal language model learns next-token prediction at every position.

For example:

```text
capital → predict what follows "capital"

capital of → predict what follows "of"

capital of france → predict what follows "france"

...

capital of france is paris → predict what follows "paris"
```

So every position gets its own set of vocabulary scores.

---

## Why do we only care about the last position during generation?

When generating new text, our current sequence is already:

```text
capital of france is paris
```

We are not trying to predict what should have come after `"capital"` anymore.

We want:

> given the entire sequence so far, what should come next?

So we take only the logits belonging to the **last token position**:

```python
last_token_logits = outputs.logits[:, -1, :]

print(last_token_logits.shape)
```

We got:

```text
torch.Size([1, 151936])
```

The indexing:

```python
[:, -1, :]
```

means:

```text
:   → keep every batch item
-1  → take only the last token position
:   → keep every vocabulary score
```

So we went from:

```text
[1, 5, 151936]
```

to:

```text
[1, 151936]
```

Now we have exactly what we need for generation:

```text
one prompt
↓
151,936 possible next tokens
↓
one raw score for each
```

So for our prompt:

```text
"capital of france is paris"
```

the model is effectively saying:

```text
token 0      → some score
token 1      → some score
token 2      → some score
...
token 151935 → some score
```

Our next job is to decide **which one of those 151,936 tokens should actually be selected**.

For the first version, we will use the simplest possible strategy:

**pick whichever token has the highest logit.**


In [15]:
next_token_id = torch.argmax(last_token_logits, dim=-1)
print(next_token_id)

# choosing he token with the highest score
#torch.argmax(...) = returns the index of the largest value.
#dim=-1 = search across the last dimension, which here is the vocabulary dimension of size 151936.
#So this is asking: “Which vocabulary token got the highest score?”

print(tokenizer.decode([next_token_id]))

#my guess is it may be a newline or punctuation-like token, because your prompt already ends with "paris"
#and the model is predicting what comes after that.
# So we have now manually done the core step that generate() normally hides.

tensor([198], device='cuda:0')
['\n']


## 7. Choosing the next token

We now have:

```text
151,936 logits
```

One score for every possible token in Qwen's vocabulary.

The model has done its job. It has scored all possible next tokens.

Now **we** need to decide which token to actually generate.

For the simplest possible version, we just choose the token with the highest score.

```python
next_token_id = torch.argmax(last_token_logits, dim=-1)

print(next_token_id)
print(repr(tokenizer.decode(next_token_id)))
```

---

## What does `torch.argmax()` do?

```python
torch.argmax(last_token_logits, dim=-1)
```

means:

> find the position containing the largest value.

Remember that `last_token_logits` has shape:

```text
[1, 151936]
```

The last dimension contains:

```text
151,936 scores
```

one for each vocabulary token.

So:

```python
dim=-1
```

means:

> search across the vocabulary dimension.

Very roughly, imagine the model produced:

```text
token ID       logit

0              -3.2
1               1.4
2               0.7
3               5.1   ← highest
4              -0.2
...
```

Then:

```python
torch.argmax(...)
```

would return:

```text
3
```

because token `3` received the highest score.

We are doing exactly the same thing here, just across **151,936 tokens**.

---

## Our output

We got:

```text
tensor([198], device='cuda:0')
```

So out of every token in Qwen's vocabulary, the token with the highest score was:

```text
198
```

Notice that the result is:

```text
tensor([198])
```

rather than simply:

```text
198
```

because we still have a batch dimension.

We had one sequence in the batch, so PyTorch gives us one predicted token:

```text
[198]
```

---

## Turning the token ID back into text

The model only gave us:

```text
198
```

So we use the tokenizer again, but this time in the opposite direction:

```python
tokenizer.decode(next_token_id)
```

Our result was:

```text
'\n'
```

Token ID:

```text
198
```

therefore represents a **newline character**.

`repr()` is useful here because a newline is invisible if we print it normally.

For example:

```python
print(tokenizer.decode(next_token_id))
```

would mostly look like an empty line.

But:

```python
print(repr(tokenizer.decode(next_token_id)))
```

shows:

```text
'\n'
```

so we can clearly see what the token actually contains.

---

## Why did the model predict a newline?

Our prompt was:

```text
capital of france is paris
```

This already reads like a completed statement.

So the model decided that, given everything it has seen so far, a newline was the most likely thing to follow.

The important thing is not whether this particular prediction is interesting.

The important thing is that we have now manually performed the basic next-token prediction process.

```text
"capital of france is paris"
             ↓
          tokenize
             ↓
[65063, 315, 47587, 374, 40858]
             ↓
           Qwen
             ↓
151,936 logits for the next position
             ↓
          argmax
             ↓
            198
             ↓
          decode
             ↓
            "\n"
```

This is the core operation hidden inside high-level generation APIs such as:

```python
model.generate(...)
```

---

## One important limitation: this is greedy decoding

By using:

```python
torch.argmax(...)
```

we always select the token with the highest score.

This strategy is called **greedy decoding**.

If the model scores:

```text
"\n"    → 22%
"\n\n"  → 21%
","     → 14%
"."     → 11%
```

`argmax` will always choose:

```text
"\n"
```

even though several other tokens may also be quite likely.

Later we will implement **sampling**, where the next token can be chosen according to the model's probability distribution instead of always taking the winner.

For now, greedy decoding is useful because it keeps the generation process as simple as possible.

---

At this point we have manually generated **one token**.

But generation obviously does not stop after one token.

The next question is:

> how do we take token `198`, feed it back into the model, and generate another token without recomputing the entire prompt again?

That is where the **KV cache and decode loop** start to matter.


In [16]:
#Now we’ll use the KV cache for the first time.
#Remember: past_key_values contains the Key/Value tensors from the 5 prompt tokens we already processed.

cache = outputs.past_key_values
print(type(cache))
print(cache)


#let's inspect how many tokens the cache currently represents.

print(cache.get_seq_length())

#first real decode step using only the newly generated token

with torch.no_grad():
  decode_output=model(
      next_token_id.unsqueeze(0),
      past_key_values=cache,
      use_cache=True
      )

print(decode_output.logits.shape)
print(decode_output.past_key_values.get_seq_length())



<class 'transformers.cache_utils.DynamicCache'>
DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer])
5
torch.Size([1, 1, 151936])
6


## 8. First decode step using the KV cache

We have generated our first new token:

```text
198 → "\n"
```

Now we want the model to continue generating.

A naive way would be to take:

```text
capital of france is paris\n
```

and run the **entire sequence through the model again**.

That works, but it would repeat a lot of computation we already did for the original 5 prompt tokens.

This is exactly why LLM inference uses a **KV cache**.

---

## Getting the cache from the first forward pass

Our original model output contained:

```text
logits
past_key_values
```

So let's take the cached values:

```python
cache = outputs.past_key_values

print(type(cache))
print(cache)
```

We got:

```text
<class 'transformers.cache_utils.DynamicCache'>

DynamicCache(
    layers=[
        DynamicLayer,
        DynamicLayer,
        ...
        DynamicLayer
    ]
)
```

`DynamicCache` is Hugging Face's object for storing the **Key and Value tensors** created by the attention layers.

Remember our model architecture had:

```text
24 Transformer layers
```

and the cache shows:

```text
24 DynamicLayer objects
```

That is not a coincidence.

Each Transformer layer performs its own attention calculation, so **each layer needs its own K and V cache**.

Very roughly:

```text
Transformer layer 1  → cached K + V
Transformer layer 2  → cached K + V
Transformer layer 3  → cached K + V
...
Transformer layer 24 → cached K + V
```

We will inspect the actual tensors inside these layers later.

For now, the main idea is:

> the KV cache remembers attention information for tokens we have already processed.

---

## How many tokens are currently cached?

```python
print(cache.get_seq_length())
```

Output:

```text
5
```

That makes sense.

Our original prompt had exactly 5 tokens:

```text
"capital"
" of"
" france"
" is"
" paris"
```

During the first full forward pass, the model calculated the Keys and Values for all 5 tokens and stored them in the cache.

So before generation continues:

```text
KV cache length = 5
```

This first pass over the whole prompt is called **prefill**.

---

# Prefill vs decode

This is one of the most important ideas in LLM inference.

### Prefill

During prefill, we give the model the whole prompt:

```text
5 tokens
   ↓
model
   ↓
process all 5 tokens
   ↓
build KV cache for all 5
```

Our prefill input had shape roughly:

```text
[1, 5]
```

because we had:

```text
1 request
5 tokens
```

After prefill:

```text
cache length = 5
```

The model then predicted token:

```text
198 → "\n"
```

Now generation enters a different phase.

### Decode

During decode, we do **not** send those original 5 tokens again.

We only send the newly generated token:

```text
"\n"
```

and reuse the cached K/V information for everything that came before it.

So instead of:

```text
process tokens 1,2,3,4,5,6 again
```

we do:

```text
cached: 1,2,3,4,5
               +
process only token 6
```

That is the core idea behind KV caching.

---

## Preparing the new token

Our generated token currently looks like:

```python
next_token_id
```

with shape:

```text
[1]
```

because it contains one predicted token for our one sequence.

The model expects token inputs in this shape:

```text
[batch, sequence_length]
```

So we use:

```python
next_token_id.unsqueeze(0)
```

`unsqueeze(0)` adds one new dimension at position `0`.

So:

```text
before: [1]
after:  [1, 1]
```

The new shape means:

```text
1 request
1 new token
```

That is exactly what we want during decode.

---

## Running the first decode step

```python
with torch.no_grad():
    decode_output = model(
        input_ids=next_token_id.unsqueeze(0),
        past_key_values=cache,
        use_cache=True
    )
```

There are three important inputs here.

### `input_ids`

```python
input_ids=next_token_id.unsqueeze(0)
```

We are giving the model **only the newly generated token**.

Not the original prompt again.

### `past_key_values`

```python
past_key_values=cache
```

This gives the model the K/V information from the previous 5 tokens.

So the model effectively has access to:

```text
capital of france is paris
```

through the cache, while only computing the new token:

```text
"\n"
```

### `use_cache=True`

This tells the model:

> after processing this new token, also add its new K/V values to the cache.

Because if we want to generate another token after this, this token is now part of the past too.

---

# What came out?

Let's inspect the logits:

```python
print(decode_output.logits.shape)
```

Output:

```text
torch.Size([1, 1, 151936])
```

Compare this with our original prefill output:

```text
prefill → [1, 5, 151936]
decode  → [1, 1, 151936]
```

This difference is important.

During prefill we processed:

```text
5 tokens
```

so we got logits for all 5 positions.

During decode we processed only:

```text
1 new token
```

so we only got logits for that one position.

But for that position, we still need:

```text
151,936 scores
```

because any token in the vocabulary could potentially come next.

So:

```text
[1, 1, 151936]
```

means:

```text
1 request
1 newly processed token
151,936 possible next-token scores
```

---

## What happened to the cache?

Now let's check:

```python
print(decode_output.past_key_values.get_seq_length())
```

Output:

```text
6
```

Before the decode:

```text
cache length = 5
```

After processing our newly generated token:

```text
cache length = 6
```

So the cache grew like:

```text
Before decode:

capital | of | france | is | paris
   1      2      3      4     5

KV cache length = 5


Process:

"\n"


After decode:

capital | of | france | is | paris | \n
   1      2      3      4     5      6

KV cache length = 6
```

This is exactly what we wanted.

The model did **not recompute the original 5 tokens**.

It reused their cached K/V values and calculated new K/V values only for token 6.

---

## One subtle detail about `DynamicCache`

Hugging Face's `DynamicCache` is mutable.

That means when we pass:

```python
past_key_values=cache
```

the model can update that same cache object in place.

So after this decode call, the original:

```python
cache
```

may itself already report a sequence length of `6`.

This becomes important if we accidentally run the same decode cell twice: the cache could grow again.

When experimenting with KV cache manually, it is therefore useful to remember that the cache has **state**.

---

We have now manually done the two major phases of autoregressive LLM inference:

```text
PREFILL
whole prompt
[1, 5]
   ↓
process all prompt tokens
   ↓
KV cache = 5 tokens
   ↓
predict first new token


DECODE
one new token
[1, 1]
   +
existing KV cache
   ↓
process only the new token
   ↓
KV cache = 6 tokens
   ↓
predict another token
```

This prefill → decode split is at the center of modern LLM serving systems.

Next we can take the logits from this decode step, choose the next token again, and start turning this into an actual **generation loop**.


In [17]:
#manually choose the second generated token.

second_token_id = torch.argmax(
    decode_output.logits[:, -1, :],
    dim=-1
)

print(second_token_id)
print(tokenizer.decode(second_token_id))

print(cache.get_seq_length())

tensor([59604], device='cuda:0')
Paris
6


## 9. Choosing the second generated token

We already used the KV cache to process our first generated token:

```text
198 → "\n"
```

That decode step gave us a fresh set of logits for what should come next.

So now we do the same thing again:

```python
second_token_id = torch.argmax(
    decode_output.logits[:, -1, :],
    dim=-1
)

print(second_token_id)
print(repr(tokenizer.decode(second_token_id)))
print(cache.get_seq_length())
```

### Output

```text
tensor([59604], device='cuda:0')
'Paris'
6
```

So the model's second generated token is:

```text
59604 → "Paris"
```

The important thing here is that we are repeating the same next-token logic we used before.

First we had:

```text
prefill logits
    ↓
argmax
    ↓
198 → "\n"
```

Now we have:

```text
decode logits
    ↓
argmax
    ↓
59604 → "Paris"
```

---

## Why are we using `decode_output.logits` now?

Earlier we used:

```python
outputs.logits
```

because `outputs` came from the **prefill** pass over the original prompt.

Now we use:

```python
decode_output.logits
```

because these are the logits produced **after processing the newly generated newline token**.

So the state has moved forward.

Conceptually:

```text
Prompt:
capital of france is paris

        ↓ prefill

predict:
"\n"

        ↓ process "\n" using KV cache

predict:
"Paris"
```

Each model call gives us logits for the **next** token after whatever has already been processed.

---

## Why is the cache still length 6?

We printed:

```python
print(cache.get_seq_length())
```

and got:

```text
6
```

This is correct.

The cache currently contains:

```text
5 original prompt tokens
+
1 generated newline token
=
6 cached tokens
```

Even though we have now **predicted**:

```text
"Paris"
```

that token is not inside the KV cache yet.

This distinction is very important.

The model has only said:

> I think token `59604` should come next.

But `"Paris"` has not actually been passed through the Transformer yet.

So right now:

```text
cached:
capital | of | france | is | paris | \n

predicted but NOT cached yet:
Paris
```

The cache grows only when a token is actually processed by the model.

So if we now run another decode call with:

```python
second_token_id
```

the model will calculate K/V for `"Paris"` and the cache length will become:

```text
7
```

---

This gives us an important mental model:

```text
current token
    ↓
run through model
    ↓
its K/V gets added to cache
    ↓
model outputs logits
    ↓
choose next token
```

The **chosen next token is always one step ahead of the cache** until we feed it back through the model.

So our generation process is now:

```text
Prompt tokens
    ↓
PREFILL
    ↓
cache = 5
    ↓
predict "\n"
    ↓
DECODE "\n"
    ↓
cache = 6
    ↓
predict "Paris"
    ↓
DECODE "Paris"
    ↓
cache = 7
    ↓
predict next token
    ↓
...
```

At this point we are basically doing autoregressive generation manually.

The only annoying part is that we are repeating the same few lines by hand every time.

Next we can put this exact process into a small loop.


In [18]:
#resetting everything

with torch.no_grad():
    outputs = model(input_ids)

cache = outputs.past_key_values

next_token_id = torch.argmax(
    outputs.logits[:, -1, :],
    dim=-1
)

print(cache.get_seq_length())
print(tokenizer.decode(next_token_id))


#Now we automate the exact decode step you just did manually. looping it 5 times

generated_ids = []

for _ in range(5):
    generated_ids.append(next_token_id.item())

    with torch.no_grad():
        outputs = model(
            input_ids=next_token_id.unsqueeze(0),
            past_key_values=cache,
            use_cache=True
        )

    next_token_id = torch.argmax(
        outputs.logits[:, -1, :],
        dim=-1
    ) #selecting the highest value token and shifting prev next token id to precited token

print(tokenizer.decode(generated_ids))



5



Paris, the capital


## 10. Turning manual decoding into a generation loop

We now know how to generate tokens one at a time.

The process is always the same:

```text
take current token
      ↓
run it through the model using the KV cache
      ↓
cache grows by one token
      ↓
model gives new logits
      ↓
choose the next token
      ↓
repeat
```

Instead of manually writing this again for every token, let's put it into a loop.

But first, we reset the model state.

---

## Resetting back to the original prompt

```python
with torch.no_grad():
    outputs = model(input_ids)

cache = outputs.past_key_values

next_token_id = torch.argmax(
    outputs.logits[:, -1, :],
    dim=-1
)

print(cache.get_seq_length())
print(repr(tokenizer.decode(next_token_id)))
```

We run the original prompt through the model again because our previous experiments had already modified the KV cache.

This gives us a fresh state:

```text
original prompt = 5 tokens
KV cache length = 5
```

and `next_token_id` again becomes the model's first prediction after the prompt.

For our prompt, that token is the newline:

```text
198 → "\n"
```

If we print it without `repr()`, the newline can look like an empty line. That is why `repr()` is useful while debugging tokens.

---

# Automating decode

Now we repeat the decode process five times:

```python
generated_ids = []

for _ in range(5):
    generated_ids.append(next_token_id.item())

    with torch.no_grad():
        outputs = model(
            input_ids=next_token_id.unsqueeze(0),
            past_key_values=cache,
            use_cache=True
        )

    next_token_id = torch.argmax(
        outputs.logits[:, -1, :],
        dim=-1
    )

print(tokenizer.decode(generated_ids))
```

Output:

```text
Paris, the capital
```

There is actually a newline token at the beginning of the generated sequence, so conceptually the output starts like:

```text
"\nParis, the capital"
```

The leading newline is just easy to miss when it gets printed normally.

---

## Understanding the loop properly

This line:

```python
generated_ids.append(next_token_id.item())
```

stores the token that has already been predicted.

For the first iteration that is:

```text
198 → "\n"
```

Then:

```python
outputs = model(
    input_ids=next_token_id.unsqueeze(0),
    past_key_values=cache,
    use_cache=True
)
```

actually processes that token through the Transformer.

This does two things:

```text
1. add this token's K/V to the cache

2. produce logits for the token that should come after it
```

Then:

```python
next_token_id = torch.argmax(
    outputs.logits[:, -1, :],
    dim=-1
)
```

chooses the next prediction.

This is the part that can initially feel confusing.

`next_token_id` is not being numerically increased or "shifted".

We are simply **replacing the variable** with the newly predicted token.

For example:

```text
Before model call:

next_token_id = 198
                   ↓
                 "\n"


Process 198 through model
                   ↓
model predicts 59604


After argmax:

next_token_id = 59604
                   ↓
                "Paris"
```

On the next loop:

```text
59604 gets stored
      ↓
"Paris" is processed
      ↓
model predicts another token
      ↓
next_token_id gets replaced again
```

So the loop moves forward like this:

```text
prefill
   ↓
predict "\n"

iteration 1
store "\n"
process "\n"
predict "Paris"

iteration 2
store "Paris"
process "Paris"
predict ","

iteration 3
store ","
process ","
predict " the"

iteration 4
store " the"
process " the"
predict " capital"

iteration 5
store " capital"
process " capital"
predict another token
```

Notice something subtle at the end.

After iteration 5, the model has already predicted a **sixth token**, but we don't store it because the loop ends.

`generated_ids` contains only the five tokens that we explicitly appended.

---

## What `generated_ids` is doing

Instead of immediately converting every token back into text, we keep their IDs:

```text
[
    198,
    59604,
    ...,
]
```

Then at the very end:

```python
tokenizer.decode(generated_ids)
```

turns the whole generated token sequence back into readable text.

So there are really two separate pieces of state here:

```text
generated_ids
    =
tokens we want to return to the user


KV cache
    =
attention information the model needs to continue generation
```

They are related, but they serve completely different purposes.

---

## We have now built autoregressive generation

At this point we are no longer just inspecting the model.

We have manually implemented the core autoregressive generation process:

```text
PROMPT
  ↓
tokenize
  ↓
PREFILL
process entire prompt
  ↓
build KV cache
  ↓
predict first token
  ↓

DECODE LOOP
  ↓
store predicted token
  ↓
process only that token
  ↓
extend KV cache
  ↓
predict next token
  ↓
repeat
```

This is the fundamental loop underneath LLM text generation.

A production inference engine has much more around it: batching, scheduling, memory management, request state, stopping conditions, and optimized kernels. But the basic generation dependency is still:

```text
predict token
    ↓
feed token back in
    ↓
predict next token
    ↓
feed it back in
    ↓
...
```

Right now we are always using:

```python
torch.argmax(...)
```

which means we always choose the highest-scoring token.



In [19]:
layer0 = cache.layers[0]
print(layer0.keys.shape)
print(layer0.values.shape)
print(layer0.keys)
print(layer0.values)


#let's inspect Qwen's exact head configuration:

print("Query heads:", model.config.num_attention_heads)
print("KV heads:", model.config.num_key_value_heads)
print("Hidden size:", model.config.hidden_size)
head_dim = model.config.hidden_size // model.config.num_attention_heads

print("Head Dim: ",head_dim)


#let’s see how much memory this KV cache is actually using.

k = layer0.keys
v = layer0.values

print("K elements:", k.numel())
print("V elements:", v.numel())
print("Bytes per value:", k.element_size())

total_bytes = 0

for layer in cache.layers:
    total_bytes += layer.keys.numel() * layer.keys.element_size()
    total_bytes += layer.values.numel() * layer.values.element_size()

print(total_bytes)
print(total_bytes / 1024, "KB")

#Let’s verify the memory per token directly from the cache:

seq_len = cache.get_seq_length()

bytes_per_token = total_bytes / seq_len

print("Sequence length:", seq_len)
print("KV bytes per token:", bytes_per_token)
print("KV KB per token:", bytes_per_token / 1024)

print("EOS token:", tokenizer.eos_token)
print("EOS token ID:", tokenizer.eos_token_id)

torch.Size([1, 2, 10, 64])
torch.Size([1, 2, 10, 64])
tensor([[[[  -8.3359,   -2.7754,   -6.1797,  ...,   34.0625, -129.6250,
             69.9375],
          [ -10.1953,   -8.2500,   -7.3281,  ...,   39.5938, -130.3750,
             67.9375],
          [  -3.1230,   -9.6641,   -6.9922,  ...,   33.8750, -129.6250,
             69.9375],
          ...,
          [ -10.8828,    9.0547,    5.5312,  ...,   39.0625, -130.3750,
             67.2500],
          [  -4.6562,    5.5938,    7.1328,  ...,   40.0625, -130.3750,
             68.0000],
          [   4.8672,    1.0391,    7.2773,  ...,   35.1562, -130.0000,
             69.2500]],

         [[ -10.3828,    0.3792,    5.9609,  ...,   60.5938,  112.0625,
           -124.3750],
          [  33.6562,   -8.7422,    7.3242,  ...,   65.3750,  117.0000,
           -119.8750],
          [  50.5000,  -15.5547,    4.0352,  ...,   60.6562,  111.8750,
           -124.8750],
          ...,
          [  21.5938,   15.7109,   -7.5742,  ...,   66.8125

## 11. Inspecting what is actually inside the KV cache

So far we have been treating the KV cache like one object that somehow "remembers" previous tokens.

Now let's actually open it up and see what is stored inside.

After our 5-token decode loop, the cache contains:

```text
5 prompt tokens
+
5 generated tokens
=
10 processed tokens
```

That is why our current cache length is `10`.

---

## Looking at one Transformer layer

Remember that Qwen has **24 Transformer layers**, and every layer has its own K and V cache.

Let's inspect only the first layer:

```python
layer0 = cache.layers[0]

print(layer0.keys.shape)
print(layer0.values.shape)

print(layer0.keys)
print(layer0.values)
```

Output:

```text
torch.Size([1, 2, 10, 64])
torch.Size([1, 2, 10, 64])
```

Both K and V have the same shape:

```text
[1, 2, 10, 64]
```

This shape is extremely important.

It means:

```text
[batch, KV heads, cached tokens, head dimension]

[1,     2,        10,            64]
```

So:

```text
1  = one request

2  = two KV heads

10 = ten tokens currently stored in the cache

64 = each head stores 64 numbers per token
```

Another way to picture just the K cache for this layer:

```text
KV head 1
    token 1  → 64 numbers
    token 2  → 64 numbers
    ...
    token 10 → 64 numbers

KV head 2
    token 1  → 64 numbers
    token 2  → 64 numbers
    ...
    token 10 → 64 numbers
```

And the V cache has the same structure.

---

## What are all those numbers?

When we printed:

```python
print(layer0.keys)
print(layer0.values)
```

we got large tensors full of values such as:

```text
-8.3359
34.0625
-129.6250
...
```

for K and different values for V.

We do not need to interpret the individual numbers.

They are intermediate representations produced by the attention projections.

Earlier in the model architecture we saw:

```text
k_proj
v_proj
```

Those layers take the hidden representation of a token and produce its **Key** and **Value** vectors.

During generation, instead of recalculating those vectors for old tokens again and again, we keep them here.

That is literally the **KV cache**.

So:

```text
token hidden state
      ↓
k_proj / v_proj
      ↓
K vector + V vector
      ↓
store them in KV cache
```

---

# Why only 2 KV heads?

Let's inspect the model configuration:

```python
print("Query heads:", model.config.num_attention_heads)
print("KV heads:", model.config.num_key_value_heads)
print("Hidden size:", model.config.hidden_size)

head_dim = (
    model.config.hidden_size
    // model.config.num_attention_heads
)

print("Head Dim:", head_dim)
```

Output:

```text
Query heads: 14
KV heads: 2
Hidden size: 896
Head Dim: 64
```

So Qwen has:

```text
14 Query heads
2 Key heads
2 Value heads
```

This is different from normal multi-head attention where we could have the same number of Q, K and V heads.

Qwen uses something called **Grouped Query Attention (GQA)**.

---

## First, where does `64` come from?

The model hidden size is:

```text
896
```

and there are:

```text
14 Query heads
```

So each Query head gets:

```text
896 / 14 = 64
```

numbers.

That gives us:

```text
head dimension = 64
```

And this matches exactly what we saw in the cache:

```text
[1, 2, 10, 64]
              ↑
          head dimension
```

---

# Grouped Query Attention

If Qwen used normal Multi-Head Attention with 14 heads, we could have:

```text
14 Query heads
14 Key heads
14 Value heads
```

But storing K and V for 14 heads for every token would make the KV cache much larger.

Instead Qwen does:

```text
14 Query heads
2 Key heads
2 Value heads
```

Several Query heads share the same K/V head.

Very roughly:

```text
Q1  ┐
Q2  │
Q3  │
Q4  │
Q5  │ → KV head 1
Q6  │
Q7  ┘


Q8  ┐
Q9  │
Q10 │
Q11 │
Q12 │ → KV head 2
Q13 │
Q14 ┘
```

This is the basic idea behind **Grouped Query Attention**.

The exact grouping happens inside the attention implementation, but the reason is easy:

```text
fewer KV heads
      ↓
smaller KV cache
      ↓
less GPU memory
      ↓
less memory movement during decode
```

This becomes extremely important for inference.

Remember: Query vectors are needed only for the token currently being processed.

But K and V vectors from **all previous tokens must stay around** during generation.

That is why reducing KV heads saves so much memory.

---

## We can even see GQA in the model weights

Earlier the model architecture showed:

```text
q_proj: 896 → 896

k_proj: 896 → 128

v_proj: 896 → 128
```

Now those numbers make sense.

For Query:

```text
14 heads × 64 values
=
896
```

For Key:

```text
2 heads × 64 values
=
128
```

For Value:

```text
2 heads × 64 values
=
128
```

So the architecture printout was already telling us that Qwen was using fewer K/V heads.

---

# How much memory is our KV cache using?

Now let's actually calculate it.

For the first layer:

```python
k = layer0.keys
v = layer0.values

print("K elements:", k.numel())
print("V elements:", v.numel())
print("Bytes per value:", k.element_size())
```

Output:

```text
K elements: 1280
V elements: 1280
Bytes per value: 2
```

Let's verify the number of K elements ourselves.

The shape was:

```text
[1, 2, 10, 64]
```

So:

```text
1 × 2 × 10 × 64
=
1280 values
```

Exactly what PyTorch reports.

The same is true for V.

---

## Why 2 bytes per value?

Our model is using:

```text
float16
```

and FP16 stores each value using:

```text
16 bits
=
2 bytes
```

So for one layer:

```text
K:
1280 × 2 bytes
=
2560 bytes

V:
1280 × 2 bytes
=
2560 bytes
```

Together:

```text
K + V
=
5120 bytes
```

for just one Transformer layer.

But Qwen has 24 layers.

---

# Calculating the entire KV cache

```python
total_bytes = 0

for layer in cache.layers:
    total_bytes += (
        layer.keys.numel()
        * layer.keys.element_size()
    )

    total_bytes += (
        layer.values.numel()
        * layer.values.element_size()
    )

print(total_bytes)
print(total_bytes / 1024, "KB")
```

Output:

```text
122880
120.0 KB
```

So for our current sequence of only **10 tokens**, the KV cache already uses:

```text
120 KB
```

This model is tiny, so that number looks small.

But the important part is how this memory grows.

---

# Memory per token

Let's calculate that directly:

```python
seq_len = cache.get_seq_length()

bytes_per_token = total_bytes / seq_len

print("Sequence length:", seq_len)
print("KV bytes per token:", bytes_per_token)
print("KV KB per token:", bytes_per_token / 1024)
```

Output:

```text
Sequence length: 10
KV bytes per token: 12288.0
KV KB per token: 12.0
```

So for this Qwen model:

```text
each cached token
≈
12 KB of KV cache
```

We can derive this directly from the architecture too.

For one token:

```text
24 layers
×
2 KV heads
×
64 values per head
×
2 tensors (K + V)
×
2 bytes per FP16 value
```

which gives:

```text
24 × 2 × 64 × 2 × 2
=
12,288 bytes
=
12 KB
```

per token.

That formula is worth remembering:

```text
KV memory per token

≈

layers
× KV heads
× head dimension
× 2       ← K and V
× bytes per value
```

---

# Why KV cache memory becomes a serving problem

Our tiny model uses:

```text
12 KB / token
```

So roughly:

```text
1,000 tokens
≈ 12 MB

10,000 tokens
≈ 120 MB
```

for a single request.

Now imagine many users generating at the same time.

Very roughly:

```text
10 requests × 1,000 cached tokens
≈ 120 MB

100 requests × 1,000 cached tokens
≈ 1.2 GB

100 requests × 10,000 cached tokens
≈ 12 GB
```

And again, this is a **0.5B model**.

Larger production models can have much larger KV caches.

This is why inference engines care so much about:

```text
KV cache allocation
block management
paged KV cache
prefix caching
eviction
preemption
fragmentation
```

The model weights mostly stay fixed in GPU memory.

But KV cache memory changes continuously depending on:

```text
number of active requests
×
number of cached tokens per request
```

That makes KV memory management one of the central problems in LLM serving.

---

# One more thing: EOS

We also inspected:

```python
print("EOS token:", tokenizer.eos_token)
print("EOS token ID:", tokenizer.eos_token_id)
```

Output:

```text
EOS token: <|im_end|>
EOS token ID: 151645
```

**EOS** means:

```text
End Of Sequence
```

It is just another token in the vocabulary.

For Qwen:

```text
151645 → <|im_end|>
```

During generation, the model may eventually predict this token.

A generation runtime can then do something like:

```text
if predicted token == EOS:
    stop generating this request
```

So the model itself does not magically terminate a Python loop.

It predicts an EOS token, and the **inference runtime decides what to do with it**.

This distinction will matter much more once we have multiple requests being scheduled together.

---

At this point we have gone from thinking of the KV cache as:

```text
"some saved attention stuff"
```

to knowing its actual structure:

```text
24 layers

each layer:
    K → [batch, KV heads, tokens, head_dim]
    V → [batch, KV heads, tokens, head_dim]

for our model:
    [1, 2, 10, 64]
```

and we now know that the cache grows **linearly with sequence length**.

That is the first real connection between basic Transformer inference and the memory-management problems that engines like vLLM are built to solve.


**In Next Chapter we will learn about replacing argmax() with softmax and the concepts of sampling in inference**